# StirrupAgent leaderboard charts

Reads the CSV written by `benchmarks/consolidate_results.py`, so the charts and
the leaderboard can never disagree:

```bash
python benchmarks/consolidate_results.py "$LEADERBOARD_DIR" --csv results.csv
```

One chart per cell, each saved to `figs/` as PNG and PDF. No horizontal
gridlines.

**What the consolidator gives us** (one row per model):
`model`, `n`, `passed`, `pass_rate`, `score_avg`, `turns_avg`, `calls_avg`,
`tokens_in_avg`, `tokens_out_avg`, `cost_total`, `cost_per_pass`.

**What it does not**: wall-clock duration, Macro-F1, and the input/output cost
split. Supply those in `EXTRAS` below and the matching panels appear; leave them
empty and those panels are skipped rather than faked.

Two things worth knowing about the columns:

- `tokens_in_avg` and `tokens_out_avg` are **per scenario**. The leaderboard
  slide shows run totals, so the notebook derives `avg x n` rather than asking
  you to retype them.
- `cost_total` is blank whenever the harness reported no `est_cost_usd`. Cost
  panels are skipped in that case instead of plotting zeros.

## Setup

In [ ]:
from pathlib import Path

import make_charts as mc          # the loader and the chart function live here

%matplotlib inline

OUTDIR  = Path("figs"); OUTDIR.mkdir(exist_ok=True)
CSV     = Path("results.csv")     # <- output of consolidate_results.py --csv
PALETTE = mc.PALETTES["default"]  # "cb" for colourblind-safe, "carbon" for IBM

## Display names and the metrics the consolidator does not emit

Raw model ids are long. Map them to what should sit under the bars; anything not
listed is shortened automatically (`anthropic/claude-opus-5-high` becomes
`claude-opus-5` with effort `high`).

In [ ]:
mc.DISPLAY.update({
    "anthropic/claude-opus-5-high":  ("Opus 5",     "high"),
    "openai/gpt-5.6-sol-max":        ("GPT-5.6",    "max"),
    "google/gemini-3.6-flash-high":  ("Gemini 3.6", "high"),
    "anthropic/claude-sonnet-5-max": ("Sonnet 5",   "max"),
    "minimax/minimax-m3-high":       ("MiniMax",    "high"),
})

# keyed by DISPLAY name. Comment a block out and its panel disappears.
mc.EXTRAS.update({
    "duration_p50": {"Opus 5": 92, "GPT-5.6": 50, "Gemini 3.6": 171,
                     "Sonnet 5": 441, "MiniMax": 110},          # seconds
    "macro_f1":     {"Opus 5": 0.468, "GPT-5.6": 0.460, "Gemini 3.6": 0.459,
                     "Sonnet 5": 0.402, "MiniMax": 0.400},
    "input_cost":   {"Opus 5": 94.20, "GPT-5.6": 34.74, "Gemini 3.6": 48.46,
                     "Sonnet 5": 72.61, "MiniMax": 5.98},
    "output_cost":  {"Opus 5": 8.69, "GPT-5.6": 3.85, "Gemini 3.6": 4.85,
                     "Sonnet 5": 23.10, "MiniMax": 0.50},
})

# left-to-right order on every chart, and the order colours are assigned in
ORDER = ["anthropic/claude-opus-5-high", "openai/gpt-5.6-sol-max",
         "google/gemini-3.6-flash-high", "anthropic/claude-sonnet-5-max",
         "minimax/minimax-m3-high"]

## Load

In [ ]:
res = mc.load_csv(CSV, ORDER) if CSV.exists() else mc.load_fallback()

print(f"{res.scenarios} scenarios, {len(res.ids)} models\n")
for m in res.ids:
    print(f"  {res.name[m]:<12} pass {res.pass_rate.get(m, 0):6.1%}"
          f"   tokens in {res.tokens_in_m.get(m, 0):5.1f}M"
          f"   total ${res.cost_total.get(m, 0):8,.2f}"
          f"   $/pass ${res.cost_per_pass.get(m, 0):6.2f}")

available = mc.panels(res)
print("\npanels with data:", ", ".join(available))

## One helper

`chart("pass_rate")` renders that panel with the defaults; pass keyword
arguments to override any of them, e.g. `chart("duration", ymax=520)`.

In [ ]:
def chart(name, **override):
    spec = mc.panels(res)[name]
    return mc.bar_chart(res, name=name, outdir=OUTDIR, palette=PALETTE,
                        close=False, **{**spec, **override})

## Quality

In [ ]:
chart("pass_rate");

In [ ]:
chart("macro_f1");

In [ ]:
chart("score_avg");

## Consumption

In [ ]:
chart("tokens_in");

In [ ]:
chart("tokens_out");

In [ ]:
chart("tool_calls");

In [ ]:
chart("turns");

In [ ]:
chart("duration");

## Cost

In [ ]:
chart("cost_total");

In [ ]:
chart("cost_per_scenario");

The panel the observations argue from: the cheapest model per token is not the
cheapest per usable answer, and Sonnet lands above Opus despite cheaper token
rates.

In [ ]:
chart("cost_per_pass");

## Spend table

In [ ]:
mc.cost_table(res, OUTDIR, close=False);

## Files written

In [ ]:
for f in sorted(OUTDIR.iterdir()):
    print(f)